# 3D Kooshball MPnRAGE on the BrainWeb phantom - one scan, per-echo contrast evolution

Same pipeline as `3D_Kooshball_GRE.ipynb` (cyclic 8-channel head coil +
isochromat/Bloch spin simulation + NUFFT preview & iterative NUFFT-SENSE CG),
with the 3D radial (golden-angle) **MPnRAGE** sequence: per segment
`180deg inversion - TI_start - N_readout spoiled GRE echoes (TI_inter apart) - TD`,
repeated `N_segments` times.

**One scan covers every inversion time.** The p-th echo of every segment fires
`TI_eff(p) = TI_start + p (TR + TI_inter)` after that segment's inversion, so
each echo index is a whole 3D volume at a different TI. Reconstructing one image
per echo index (the "frames" of `2D_Radial_MPnRAGE.ipynb`) plots the brain
multiple times and shows the contrast change as the magnetization recovers:
WM nulls first, then GM, then CSF. (Averaging all echoes of a train into one
image - as the per-tissue "null point" version did - washes the null out,
because the effective TI spans ~3 s and every tissue has recovered by the
later echoes.)

**Which TI nulls a tissue?** The inversion fires every segment (not after full
relaxation), so the magnetization seen by the inversion pulse is the *cyclic*
leftover, not +M0. With echo spacing `P = TR + TI_inter`:

    Mz_0(TI)    = M0 [ 1 - (1 + w) E_TI ],   w = cyclic state before the inversion
    null point  =>  E_TI = 1 / (1 + w),  and at the null w = z* (TI-independent)
    z*          = (1 - E_tot) + E_tot cos(a) kap (1 - lam^(N-1))
    lam = exp(-P/T1) cos(a),  kap = (1 - exp(-P/T1)) / (1 - exp(-P/T1) cos(a))
    E_tot = exp(-(TR+TD)/T1),  E_TD = exp(-TD/T1),  N = N_readout

(verified against the isochromat simulator: the first echo of a converged
segment is ~0 exactly at these TIs)

**Runtime on an A100** (defaults below): ~2 min null check, ~40-50 min for the
single 6000-spoke simulation (the same cost as one of the old per-tissue
scans), a few min for the per-frame NUFFT previews + CG recon + plots. DeepADMM
training is optional (~15-40 min).


In [ ]:
%matplotlib widget

import os
import sys
import time
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import itertools
import numpy as np
import pypulseq as pp
from pypulseq import Sequence
import torch
import torch.nn as nn
import torchkbnufft as tkbn
from Modules.CoilSens import cyclic_head_coil
from Modules.FourierTransform import (
    nonuniform_fourier_transform_adjoint,
    nonuniform_fourier_transform_forward,
)
from Modules.Linops import ChannelOperator, LinearOperator
from Modules.NumSolvers import conjugate_gradient
from Modules.Regularizer import Regularizer
from Modules.fast_iso import isochromat_sim_fast
from Modules.Plan import KbNufft, KbNufftAdjoint
from tqdm import tqdm

sys.path.insert(0, "..")
from Modules import default_system, load_brain_phantom

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
def build_mpnrage(
    FOV: float = 250e-3,
    N_x: int = 128,
    alpha: float = 5.0,
    N_channels = 8,
    slice_thickness: float = 3e-3,
    TE: float = 3.98e-3,
    N_readout: int = 30,
    N_segments: int = 200,
    N_dummy: int = 20,
    TI_start: float = 20e-3,
    TI_inter: float = 100e-3,
    TD: float = 500e-3,
    rf_spoiling_inc: float = 117.0,
    write_seq: bool = False,
    name: str = "MPnRAGE_Kooshball_TI20ms.seq",
) -> pp.Sequence:

    system = default_system()
    seq = pp.Sequence(system)
    deltak = 1 / FOV

    rf_inv, gz_inv, _ = pp.make_sinc_pulse(
        flip_angle=np.pi, duration=10e-3, slice_thickness=slice_thickness,
        apodization=0.5, time_bw_product=4, system=system,
        return_gz=True, use='inversion')

    rf, gz, _ = pp.make_sinc_pulse(
        apodization=0.5, duration=2e-3, flip_angle=alpha*np.pi/180,
        slice_thickness=slice_thickness, system=system,
        time_bw_product=4, return_gz=True, use='excitation')

    gx_base = pp.make_trapezoid(channel='x', flat_area=N_x*deltak, flat_time=6.4e-3/5, system=system)
    adc = pp.make_adc(num_samples=N_x, duration=gx_base.flat_time, delay=gx_base.rise_time, system=system)
    gx_pre_base = pp.make_trapezoid(channel='x', area=-gx_base.area/2, duration=2e-3, system=system)
    gz_reph = pp.make_trapezoid(channel='z', area=-gz.area/2, duration=2e-3, system=system)
    gx_spoil_base = pp.make_trapezoid(channel='x', area=0.5*N_x*deltak, system=system)
    gz_spoil = pp.make_trapezoid(channel='z', area=4/slice_thickness, system=system)

    delay_TE = (
        np.ceil((
            TE - pp.calc_duration(gx_pre_base)
            - gz.fall_time - gz.flat_time/2
            - pp.calc_duration(gx_base)/2
        ) / seq.grad_raster_time) * seq.grad_raster_time
    )
    inv_duration   = pp.calc_duration(rf_inv, gz_inv)
    delay_TI_start = (
        np.ceil((TI_start - inv_duration/2 - pp.calc_duration(gz)/2)
                / seq.grad_raster_time) * seq.grad_raster_time
    )
    assert delay_TI_start > 0, 'TI_start too short'
    delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

    gre_block_dur = (
        pp.calc_duration(gz) + pp.calc_duration(gx_pre_base)
        + delay_TE + pp.calc_duration(gx_base)
        + pp.calc_duration(gx_spoil_base, gz_spoil)
    )
    delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time

    total_acquired = N_segments * N_readout
    def spoke_dir(idx):
        phi = np.arccos(1 - 2*(idx+0.5)/total_acquired)
        theta = np.pi*(1+np.sqrt(5))*idx
        return np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi)

    rf_phase = 0.0; rf_inc = 0.0; spoke_counter = 0

    for seg in range(N_segments):
        seq.add_block(rf_inv, gz_inv)
        seq.add_block(pp.make_delay(delay_TI_start))

        n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
        for s in range(n_this_seg):
            is_dummy = (seg == 0) and (s < N_dummy)

            rf.phase_offset  = rf_phase / 180 * np.pi
            adc.phase_offset = rf_phase / 180 * np.pi
            rf_inc   = (rf_inc + rf_spoiling_inc) % 360
            rf_phase = (rf_phase + rf_inc) % 360

            kx, ky, kz = (1.0, 0.0, 0.0) if is_dummy else spoke_dir(spoke_counter)
            if not is_dummy:
                spoke_counter += 1

            seq.add_block(rf, gz)
            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_pre_base.area*kx, duration=2e-3, system=system),
                pp.make_trapezoid(channel='y', area=gx_pre_base.area*ky, duration=2e-3, system=system),
                pp.make_trapezoid(channel='z', area=gx_pre_base.area*kz+gz_reph.area, duration=2e-3, system=system),
            )
            seq.add_block(pp.make_delay(delay_TE))

            gx = pp.make_trapezoid(channel='x', amplitude=gx_base.amplitude*kx,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
            gy = pp.make_trapezoid(channel='y', amplitude=gx_base.amplitude*ky,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
            gz_ro = pp.make_trapezoid(channel='z', amplitude=gx_base.amplitude*kz,
                        flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)

            if is_dummy: seq.add_block(gx, gy, gz_ro)
            else: seq.add_block(gx, gy, gz_ro, adc)

            seq.add_block(
                pp.make_trapezoid(channel='x', area=gx_spoil_base.area*kx, system=system),
                pp.make_trapezoid(channel='y', area=gx_spoil_base.area*ky, system=system),
                pp.make_trapezoid(channel='z', area=gx_spoil_base.area*kz+gz_spoil.area, system=system),
            )
            # recovery between echoes: this is what makes each echo a different
            # effective TI (MPnRAGE)
            if s < n_this_seg - 1:
                seq.add_block(pp.make_delay(delay_TI_inter))

        seq.add_block(pp.make_delay(delay_TD))

    ok, err = seq.check_timing()
    if ok:
        print('MPnRAGE timing check passed')
        print(f'  GRE block dur : {gre_block_dur*1000:.1f} ms')
        print(f'  Echo spacing P: {(gre_block_dur+delay_TI_inter)*1000:.1f} ms')
        print(f'  Readout train : {N_readout*(gre_block_dur+delay_TI_inter)*1000:.0f} ms')
        est = N_segments*(inv_duration+delay_TI_start+N_readout*gre_block_dur
                          +(N_readout-1)*delay_TI_inter+TD)
        print(f'  Scan time est : {est/60:.1f} min')
    else: print(f'Timing FAILED: {err}')

    seq.set_definition('FOV',  [FOV, FOV, slice_thickness])
    seq.set_definition('Name', name.replace('.seq', ''))
    if write_seq: seq.write(name)
    return seq, gre_block_dur, delay_TI_inter, TD

In [ ]:
device = torch.device("cuda")

# ---- single source of truth ------------------------------------------------
FOV        = 250e-3
n          = 64     # phantom + recon matrix (n^3)
N_x        = n         # readout samples/spoke -> kmax matches the recon grid
N_readout  = 20      # echoes per inversion segment (each echo index = one image)
N_segments = 750       # inversion segments (200*30 = 6000 spokes)
N_dummy    = 20
TI_START   = 20e-3     # TI of the first echo; the train then spans ~20 ms -> 3.1 s
TI_inter   = 100e-3    # recovery delay between echoes (echo spacing P = TR + TI_inter)
N_channels = 8
spin_count = 64        # isochromats per voxel

# Tissue parameters (exactly as defined in Modules/phantom.py):
#   label 1 -> CSF: PD 1.0, T1 4.0 s, T2 0.5 s
#   label 2 -> GM : PD 0.8, T1 1.3 s, T2 0.08 s
#   label 3 -> WM : PD 0.7, T1 0.9 s, T2 0.07 s
#   T2* = 0.04 s everywhere
TISSUES = {
    "CSF": dict(T1=4.0, T2=0.5,  PD=1.0, pd_range=(0.9,  1.01)),
    "GM":  dict(T1=1.3, T2=0.08, PD=0.8, pd_range=(0.75, 0.9)),
    "WM":  dict(T1=0.9, T2=0.07, PD=0.7, pd_range=(0.1,  0.75)),
}
T2STAR = 0.04
NAMES = ("WM", "GM", "CSF")

# ---- phantom (BrainWeb) ----------------------------------------------------
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n, n, n),
    device=device,
    permute_order=(2, 1, 0),
)
print(type(phantom).__name__, "size =", phantom.size)

# tissue masks on the same grid as the phantom (same mapping as phantom.py)
raw = np.fromfile("subject04_crisp_v.rawb", dtype=np.uint8).reshape(362, 434, 362)
pd_sig = np.zeros_like(raw, dtype=np.float32)
pd_sig[raw == 1] = 1.0
pd_sig[raw == 2] = 0.8
pd_sig[raw == 3] = 0.7
pd_map = torch.nn.functional.interpolate(
    torch.from_numpy(pd_sig).to(device).view(1, 1, 362, 434, 362),
    size=(n, n, n), mode="trilinear").squeeze().permute(2, 1, 0)
masks = {}
for tname in NAMES:
    lo, hi = TISSUES[tname]["pd_range"]
    masks[tname] = ((pd_map > lo) & (pd_map < hi)).to(torch.float32).cpu().numpy()
    print(f"mask {tname}: {masks[tname].sum():.0f} voxels")
# the recon operator uses k with flipped sign, so the image is the phantom
# inverted through the origin:
masks_img = {t: np.ascontiguousarray(np.flip(m, (0, 1, 2))) for t, m in masks.items()}

# ---- coils -----------------------------------------------------------------
scale = tuple(float(s) / FOV for s in phantom.size)
coil_sim = cyclic_head_coil((N_channels, n, n, n), fov_scaling=scale).to(torch.complex64)
phantom.coil_sens = coil_sim.to(device)
data = phantom.build()
print("simulation voxels:", data.PD.numel())

In [ ]:
# ---- null-point TIs ---------------------------------------------------------
# Build once (any TI) to get the exact timings: TR = GRE block duration,
# P = TR + inter-echo delay. The timings do not depend on TI.
_, TR_echo, delay_TI_inter, TD = build_mpnrage(TI_start=100e-3)
P_echo = TR_echo + delay_TI_inter
alpha = 5.0 * np.pi / 180

def null_ti(T1, P, TR=TR_echo, TD=TD, alpha=alpha, N=N_readout):
    # TI at which the first excitation of each (cyclic) segment sees Mz = 0.
    # Cyclic state: the magnetization just before the inversion pulse is the
    # fixed point of one segment  (inversion -> TI -> N echoes -> TD).
    # At the null (Mz0 = 0) that fixed point is TI-independent:
    #     z* = (1 - E_tot) + E_tot cos(a) Mz_ss (1 - lam^(N-1))
    #     E_tot = exp(-(TR_r + TD)/T1),  lam = exp(-P/T1) cos(a)
    #     Mz_ss = (1 - E1) / (1 - E1 cos(a)),  E1 = exp(-P/T1)
    # and the null condition (1 - 2 E_TI + E_TI E_TD (1 - kap)) = 0 gives
    # E_TI = 1/(1 + z*).
    E1 = np.exp(-P / T1)
    lam = E1 * np.cos(alpha)
    kap = (1 - E1) / (1 - E1 * np.cos(alpha))          # Mz_ss / M0
    E_tot = np.exp(-(TR + TD) / T1)
    z_star = (1 - E_tot) + E_tot * np.cos(alpha) * kap * (1 - lam ** (N - 1))
    return T1 * np.log(1.0 + z_star)

TI_null = {t: null_ti(p["T1"], P_echo) for t, p in TISSUES.items()}
ti_effs = np.array([TI_START + p * P_echo for p in range(N_readout)])
null_frame = {t: int(np.argmin(np.abs(ti_effs - TI_null[t]))) for t in TI_null}
for t in NAMES:
    print(f"TI_null({t}) = {TI_null[t]*1e3:7.1f} ms   frame {null_frame[t]:2d} "
          f"(TI_eff = {ti_effs[null_frame[t]]*1e3:5.0f} ms)   "
          f"(T1*ln2 = {TISSUES[t]['T1']*np.log(2)*1e3:7.1f} ms)")


In [ ]:
# ---- quick check: first echo at the null point is ~0 ------------------------
# 4-segment 32^3 single-tissue sim (3 warm-up segments + 1 data segment, so
# the data segment is on the cyclic state - needed for the long-T1 CSF);
# compare the magnitude of the FIRST acquired echo at TI_null vs a longer TI.
def mini_first_echo(T1, T2, PD, TI, Nmini=32, Nseg=4, Nsp=N_readout):
    shape = (Nmini, Nmini, Nmini)
    dv = 250.0 / Nmini
    aff = torch.eye(4, device=device)
    for i in range(3):
        aff[i, i] = dv
        aff[i, 3] = -dv * (Nmini // 2)
    ph = mr0.VoxelGridPhantom(
        torch.full(shape, PD, device=device), torch.full(shape, T1, device=device),
        torch.full(shape, T2, device=device), torch.full(shape, T2STAR, device=device),
        torch.zeros(shape, device=device), torch.zeros(shape, device=device),
        torch.ones((1,) + shape, device=device), torch.ones((1,) + shape, device=device), aff)
    ph.coil_sens = torch.ones((8,) + shape, dtype=torch.complex64, device=device)
    d = ph.build()
    seq, _, _, _ = build_mpnrage(TI_start=TI, N_segments=Nseg, N_readout=Nsp,
                              write_seq=True, name=f"/tmp/mpnrage_check_{int(TI*1000)}.seq")
    m = mr0.Sequence.import_file(f"/tmp/mpnrage_check_{int(TI*1000)}.seq", exact_trajectories=False)
    torch.manual_seed(0)
    sig = isochromat_sim_fast(m, d, spin_count=spin_count, spin_dist="rand",
                              print_progress=False)
    sig = sig.permute(1, 0).contiguous()
    n_adc = sig.shape[1] // (Nseg * Nsp)          # samples per spoke
    first = sig[:, -Nsp * n_adc:].sum(0)[:n_adc]  # first spoke of last segment
    return first.abs().mean().item()

for t in NAMES:
    p = TISSUES[t]
    a = mini_first_echo(p["T1"], p["T2"], p["PD"], TI_null[t])
    b = mini_first_echo(p["T1"], p["T2"], p["PD"], TI_null[t] + 300e-3)
    print(f"{t}: |1st echo| at TI_null = {a:.5f}   at TI_null+300ms = {b:.5f}   "
          f"(ratio {b/(a+1e-12):.1f}x)")


In [ ]:
import os
for f in ["MPnRAGE_kooshball3D_signal.pt", "MPnRAGE_Kooshball_TI20ms.mr0cache.pt"]:
    if os.path.exists(f): os.remove(f)

In [ ]:
# ---- simulate the one MPnRAGE scan ------------------------------------------
# TI_start = 20 ms: the 30-echo train then spans TI_eff ~ 20 ms -> 3.1 s, which
# covers all three tissue nulls, so a single simulation gives every frame.
TI = TI_START
name     = f"MPnRAGE_Kooshball_TI{int(round(TI*1000))}ms_n{N_x}_R{N_readout}_S{N_segments}.seq"
sig_file = f"MPnRAGE_kooshball3D_signal_n{N_x}_R{N_readout}_S{N_segments}.pt"
build_mpnrage(FOV=FOV, N_x=N_x, N_readout=N_readout, N_segments=N_segments,
              TI_start=TI, write_seq=True, name=name)
cache = name.replace(".seq", ".mr0cache.pt")
try:
    mr_seq = torch.load(cache, weights_only=False)
    print(f"loaded {name} from cache")
except Exception:
    t1 = time.time()
    mr_seq = mr0.Sequence.import_file(name, exact_trajectories=False)
    torch.save(mr_seq, cache)
    print(f"imported {name}: {time.time()-t1:.0f} s (cached)")

n_adc = N_segments * N_readout * N_x   # dummies acquire no ADC samples
sig_file = "MPnRAGE_kooshball3D_signal.pt"
if os.path.exists(sig_file):
    signal = torch.load(sig_file)
    print(f"loaded saved signal {sig_file}  shape {tuple(signal.shape)}")
else:
    torch.manual_seed(0)
    t1 = time.time()
    signal = isochromat_sim_fast(mr_seq, data.cuda(), spin_count=spin_count,
                                 spin_dist="rand", print_progress=True,
                                 chunk_bytes=2**30)
    print(f"simulation: {time.time()-t1:.0f} s")
    torch.save(signal.permute((1, 0)).contiguous().to(torch.complex64).cpu(), sig_file)
# the simulator outputs (N_adc, N_channels); the recon pipeline wants (N_channels, N_adc)
if signal.shape[0] != N_channels:
    signal = signal.permute((1, 0))
signal = signal.to(device).contiguous().to(torch.complex64)
assert tuple(signal.shape) == (N_channels, n_adc), (tuple(signal.shape), (N_channels, n_adc))
print("simulated signal:", tuple(signal.shape))

In [ ]:
def kspace_to_rad(mr_seq, n, FOV):
    'ADC k-space (1/m) -> rad/pixel of an n^3 grid spanning FOV. (3, N_adc)'
    k = torch.as_tensor(mr_seq.get_kspace()).float()
    k = k.reshape(-1, k.shape[-1])[:, :3].T.contiguous()
    return k * (2 * np.pi * FOV / n)


def nufft_recon(signal, k_rad, n):
    'Per-channel density-compensated adjoint NUFFT. signal: (C, N) -> (C, n, n, n)'
    k_rad = k_rad.to(signal.device)
    keep = k_rad.norm(dim=0) <= np.pi                 # drop (don't clamp!) samples beyond the grid's kmax
    k, s = k_rad[:, keep], signal[:, keep]
    print(f"using {int(keep.sum())}/{keep.numel()} samples")
    dcomp = k.norm(dim=0).clamp(min=np.pi / n) ** 2   # |k|^2 for radial
    adj = tkbn.KbNufftAdjoint(im_size=(n, n, n), grid_size=(2 * n,) * 3).to(signal.device)
    return adj((s * dcomp)[None], k)[0]


def _rel_profile(a):
    a = np.abs(a)
    return a / (np.sqrt((a ** 2).sum(0, keepdims=True)) + 1e-12)


def match_coil_orientation(coil, img, mask_frac=0.3):
    # Which of the 48 axis permutations/flips of coil (C,n,n,n) matches the
    # orientation of the raw NUFFT images img (C,n,n,n)?  Compares
    # |coil_c|/rss(coil) with |img_c|/rss(img) inside the head.
    coil, img = np.asarray(coil), np.asarray(img)
    rss = np.sqrt((np.abs(img) ** 2).sum(0))
    mask = rss > mask_frac * rss.max()
    ref = _rel_profile(img)[:, mask].ravel(); ref = ref - ref.mean()
    best = (-2.0, None, None)
    for perm in itertools.permutations((0, 1, 2)):
        cp = coil.transpose(0, *[p + 1 for p in perm])
        for flips in itertools.product([False, True], repeat=3):
            dims = tuple(d + 1 for d, f in enumerate(flips) if f)
            cand = np.flip(cp, dims) if dims else cp
            v = _rel_profile(cand)[:, mask].ravel(); v = v - v.mean()
            score = float(ref @ v / (np.linalg.norm(ref) * np.linalg.norm(v) + 1e-12))
            if score > best[0]:
                best = (score, perm, flips)
    score, perm, flips = best
    dims = tuple(d + 1 for d, f in enumerate(flips) if f)
    out = coil.transpose(0, *[p + 1 for p in perm])
    return np.ascontiguousarray(np.flip(out, dims) if dims else out), perm, flips, score


In [ ]:
import inspect
print(inspect.signature(nufft_recon))

In [ ]:
# ---- one image per echo index (the "frames" of 2D_Radial_MPnRAGE.ipynb) -----
# Echo p of every segment fired TI_eff(p) after that segment's inversion, so
# each frame is the brain at a single effective TI.  Frame p uses the p-th
# acquired echo of all N_segments (N_segments spokes = 200 per frame).
k_full = kspace_to_rad(mr_seq, n, FOV)
assert k_full.shape[1] == signal.shape[1], (k_full.shape, signal.shape)

def frame_spoke_blocks(p):
    'ADC block (N_x samples) index of echo p in every segment (dummies acquire no samples)'
    return [s * N_readout + p for s in range(N_segments)]

coil_recon = None
frame_imgs = {}
for p in range(N_readout):
    idx = torch.cat([torch.arange(b * N_x, (b + 1) * N_x) for b in frame_spoke_blocks(p)])
    img = nufft_recon(signal[:, idx], k_full[:, idx], n)    # (C, n, n, n), raw NUFFT orientation
    if coil_recon is None:
        coil_np, perm, flips, score = match_coil_orientation(coil_sim.cpu().numpy(), img.cpu().numpy())
        print(f"coil orientation: perm={perm} flips={flips} score={score:.3f}")
        coil_recon = torch.from_numpy(coil_np).to(img.device)
    den = coil_recon.abs().square().sum(0)
    combined = (coil_recon.conj() * img).sum(0) / den.clamp(min=1e-2 * den.max())
    frame_imgs[p] = combined.abs().cpu().numpy()
    print(f"frame {p:2d}  TI_eff = {ti_effs[p]*1e3:7.1f} ms")


In [ ]:
# ---- the contrast evolution: one brain per echo index ------------------------
mid = n // 2
fig, axes = plt.subplots(4, 5, figsize=(16, 13))
for p, ax in enumerate(axes.ravel()):
    ax.imshow(np.rot90(frame_imgs[p][:, :, mid].T, 2), cmap="gray")
    tag = "   <- " + ",".join(t for t in NAMES if null_frame[t] == p)
    ax.set_title(f"frame {p}\nTI = {ti_effs[p]*1e3:.0f} ms{tag}", fontsize=9)
    ax.axis("off")
plt.suptitle("MPnRAGE - one scan, one image per echo index (NUFFT-adjoint previews)")
plt.tight_layout()
plt.show()


In [ ]:
# ---- per-tissue contrast vs effective TI (the nulls are the minima) ----------
fig, ax = plt.subplots(figsize=(9, 5))
for t in NAMES:
    m = masks_img[t] > 0.5
    vals = [frame_imgs[p][m].mean() for p in range(N_readout)]
    ax.plot(ti_effs * 1e3, vals, "o-", markersize=4, label=f"{t} (null {TI_null[t]*1e3:.0f} ms)")
    ax.axvline(TI_null[t] * 1e3, ls=":", alpha=0.4, color=ax.lines[-1].get_color())
ax.set_xlabel("Effective TI (ms)")
ax.set_ylabel("Mask-mean signal (reconstructed frame)")
ax.set_title("MPnRAGE magnetization recovery - per-tissue contrast vs TI")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ---- k-space trajectory for the iterative recon (same as the GRE notebook) --
# All 6000 spokes (every echo index, every segment) = the full MPnRAGE dataset.
seq = pp.Sequence()
seq.read(name)
kspace = seq.calculate_kspace()
k_traj_adc = kspace[0]
k_traj_adc = k_traj_adc[:3, :]
k_traj_radius = np.sqrt(np.sum(k_traj_adc**2, axis=0, keepdims=True))
k_traj_adc = k_traj_adc / k_traj_radius.max() * 0.5
k_traj_adc = torch.tensor(k_traj_adc, dtype=torch.float32)
print("k traj:", tuple(k_traj_adc.shape))


In [ ]:
from torch.utils.checkpoint import checkpoint
from Modules.Resnet import ResNet

class DeepADMM(nn.Module):
    """ADMM for   min  (1/2 nK) ||A x - d||^2  +  rho * g(x),   g = learned regularizer.

    Q and b passed to forward() MUST be the *normalized* data term:
        Q = (A.H @ A) / nK ,   b = (A.H @ d) / nK        (nK = # complex samples)
    For this dataset lambda_max(Q) ~ 1.2e-3, so rho ~ 1e-4..1e-3 is on the same
    scale and the regularizer actually has an effect.

    (In the old version Q was the unnormalized A.H @ A with lambda_max ~ 450, so
    rho=0.1 was ~2e-4 of the data term: the ADMM fixed point was numerically the
    plain-CG solution and the "learned" model could not move the image at all.)
    """

    def __init__(self, shape, rho_init=3e-4, maxiter_admm=1, maxiter_cg=20,
                 kernel_size=(3, 3, 3), features=16, num_of_resblocks=4,
                 scale_factor=1, checkpoint_iterations=True):
        super().__init__()
        self.rho = nn.Parameter(torch.tensor(rho_init, dtype=torch.float32))
        self.regularizer = Regularizer(
            shape, kernel_size=kernel_size, features=features,
            num_of_resblocks=num_of_resblocks, scale_factor=scale_factor,
        )
        self.I = IdentityOperator(shape)
        self.maxiter_admm = maxiter_admm
        self.maxiter_cg = maxiter_cg
        self.checkpoint_iterations = checkpoint_iterations

    def _step(self, x, z, u, Q, b, maxiter_cg):
        x = conjugate_gradient(Q + self.rho * self.I, b=b + self.rho * (z - u),
                               x=x, maxiter=maxiter_cg, disable_progressbar=True)
        z = self.regularizer(x + u)
        u = u + x - z
        return x, z, u

    def forward(self, Q, b, x, z=None, maxiter_admm=None, maxiter_cg=None,
                disable_progressbar=True):
        # warm start: x = data-fitting solution (e.g. the CG result), z = g(x)
        n_admm = self.maxiter_admm if maxiter_admm is None else maxiter_admm
        n_cg = self.maxiter_cg if maxiter_cg is None else maxiter_cg
        if z is None:
            z = self.regularizer(x)
        u = torch.zeros(self.I.shape[0], dtype=x.dtype)
        use_ckpt = self.checkpoint_iterations and self.training and torch.is_grad_enabled()
        for _ in range(n_admm):
            if use_ckpt:
                x, z, u = checkpoint(self._step, x, z, u, Q, b, n_cg, use_reentrant=False)
            else:
                x, z, u = self._step(x, z, u, Q, b, n_cg)
        return x


In [ ]:
class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities

        self.forward_shape = (1, *data_shape)
        self.adjoint_shape = (nC, *data_shape)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class NonuniformFourierTransformOperator(LinearOperator):
    'F: image (nX,nY,nZ) -> multi-coil k-space (nC, nK). k: (3, nK) in cycles/voxel.'

    def __init__(self, k, input_shape, device=None):
        nD, nK = k.shape                      # (3, nK)
        nC, nX, nY, nZ = input_shape
        self.n_modes = (nX, nY, nZ)
        self.k = (-k).unsqueeze(-1)           # (3, nK, 1), sign flipped as in the Kooshball notebooks
        self.forward_shape = (nC, nX, nY, nZ, 1)
        self.adjoint_shape = (nC, nK)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.nufft_ob = KbNufft(im_size=self.n_modes).to(device)
        self.adj_ob = KbNufftAdjoint(im_size=self.n_modes).to(device)
        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = nonuniform_fourier_transform_forward(self.k, x, nufft_ob=self.nufft_ob)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x):
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = nonuniform_fourier_transform_adjoint(
            self.k, x, self.n_modes, adj_ob=self.adj_ob
        )
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

class GradientOperator(LinearOperator):

    def __init__(
        self,
        dim: int,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.dim = dim
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.forward_shape)
        y = x - torch.roll(x, 1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.view_as_complex(x.reshape(-1, 2)).reshape(self.adjoint_shape)
        y = x - torch.roll(x, -1, self.dim)
        return torch.view_as_real(y.reshape(-1)).reshape(-1)


class IdentityOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

        self.dtype = torch.float32
        self.device = device

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        return x

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        return x


In [ ]:
# ---- standard CG (iterative NUFFT-SENSE), same as cells 9 & 10 of 3D_Kooshball_GRE.ipynb ----
F = NonuniformFourierTransformOperator(k_traj_adc, (N_channels, N_x, N_x, N_x))
C = ChannelOperator(coil_sim, (N_x, N_x, N_x))
A = F @ C

In [ ]:
# ---- CG-SENSE recon of the echo frame at each tissue's null TI ---------------
half_width = 0     # 0 = only the null echo (200 spokes); 1 = null echo +/- 1 (600 spokes), etc.
cg_iters   = 30

null_vols = {}
for t in NAMES:                                    # ("WM", "GM", "CSF")
    p0 = null_frame[t]
    ps = range(max(0, p0 - half_width), min(N_readout, p0 + half_width + 1))

    # ADC sample indices of these echo indices across all segments
    idx = torch.cat([torch.arange(b * N_x, (b + 1) * N_x)
                     for p in ps for b in frame_spoke_blocks(p)])

    F_p = NonuniformFourierTransformOperator(k_traj_adc[:, idx], (N_channels, N_x, N_x, N_x))
    A_p = F_p @ C

    d_p = signal[:, idx].contiguous().view(torch.float32).ravel()
    b_p = A_p.H @ d_p
    x0  = torch.zeros(N_x**3, dtype=torch.complex64).view(torch.float32)
    x_p = conjugate_gradient(A_p.H @ A_p, b_p, x0, maxiter=cg_iters)

    null_vols[t] = torch.abs(x_p.view(torch.complex64).reshape(N_x, N_x, N_x)).cpu().numpy()
    print(f"{t} nulled: frame {p0}, TI_eff = {ti_effs[p0]*1e3:.0f} ms, {len(idx)//N_x} spokes")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.close('all') 

In [ ]:
# ---- plot: rows = nulled tissue, columns = 3 orthogonal planes ---------------
fig, axes = plt.subplots(3, 3, figsize=(5, 5))
for r, t in enumerate(NAMES):
    vol = null_vols[t]
    vmax = np.percentile(vol, 99.5)                # one display scale per row
    views = [np.rot90(vol[N_x // 2, :, :].T, 2),
             np.rot90(vol[:, N_x // 2, :].T, 2),
             np.rot90(vol[:, :, N_x // 2].T, 2)]
    for c, (v, pname) in enumerate(zip(views, ["X-slice", "Y-slice", "Z-slice"])):
        axes[r, c].imshow(v, cmap="gray", vmin=0, vmax=vmax)
        axes[r, c].set_title(f"{t} nulled - {pname}", fontsize=10)
        axes[r, c].axis("off")
    axes[r, 0].text(-0.05, 0.5, f"{t} nulled\nTI = {ti_effs[null_frame[t]]*1e3:.0f} ms",
                    transform=axes[r, 0].transAxes, ha="right", va="center", fontsize=11)

plt.suptitle("CG-SENSE reconstruction at each tissue's null TI")
plt.tight_layout()
plt.show()

## Why the original DeepADMM looked identical to CG - and the fix

**Diagnosis** (all measured on this dataset):

1. **The regularizer's weight was ~4 orders of magnitude too small.** The ADMM x-update
   solves `(Q + rho*I) x = b + rho*(z-u)` with the *unnormalized* Gram matrix
   `Q = A^H A`. Here `lambda_max(Q) ~= 450` (750 spokes) while `rho = 0.1`, so the
   learned regularizer contributed only `rho/lambda_max ~= 2e-4` of the data term: the
   ADMM fixed point was numerically the plain-CG solution, whatever the network learned.
2. **Learnable `rho` drifted downward** (old checkpoint: 0.1 -> 0.087; with the
   normalized data term it collapses to ~1e-10 in a single step), because the
   consistency-only loss prefers a weaker regularizer.
3. **The full 750-spoke frame is over-determined** (768k real measurements vs 524k real
   unknowns, ratio 1.46): on noise-free over-determined data the least-squares solution
   is (nearly) unique, so *no* regularizer can change the answer - the old evaluation
   left no room for an "improvement".
4. **The network was trained in the broken regime** (its effect was ~0 during the
   unroll), so it never learned a useful prior; given a meaningful rho it actually
   degraded the image.
5. Bonus: the 30-iteration CG baseline is itself under-converged - radial `A^H A` is
   ill-conditioned, and its error is ~3x that of a converged CG on the full frame.

**Fix** (verified below):

- Normalize the data term: `Q = A^H A / nK`, `b = A^H d / nK` -> `lambda_max ~= 1.2e-3`
  (independent of the number of spokes); `rho = 3e-4` is then on the same scale, and is
  **fixed** as a hyperparameter.
- Pre-train the regularizer as an aliasing denoiser (~45 min incl. pair building), so it actually regularizes
  once ADMM gives it real weight.
- Evaluate at under-determined sampling (150/60 spokes), where CG carries aliasing and
   the learned prior wins; the full-frame comparison is kept as a sanity check.


In [ ]:
plt.close('all')
null_ps   = [null_frame[t] for t in NAMES]          # e.g. [5, 7, 14]
frame_idx = {p: torch.cat([torch.arange(bk * N_x, (bk + 1) * N_x)
                           for bk in frame_spoke_blocks(p)]) for p in null_ps}
C_t = ChannelOperator(coil_sim, (N_x, N_x, N_x))

def gather(idx, spokes):                              # spoke-level index selection
    return torch.cat([idx[s * N_x:(s + 1) * N_x] for s in spokes.tolist()])

# ============================================================================
# DeepADMM training (fixed). ~45 min on the GPU.
#
# Fixes relative to the original cell:
#  1. normalized data term  Q = A.H @ A / nK,  b = A.H @ d / nK
#     (lambda_max ~ 1.2e-3, independent of the number of spokes),
#  2. rho = 3e-4 and FIXED - a learnable rho collapses to ~0 under the
#     consistency-only loss (the old checkpoint drifted 0.1 -> 0.087),
#  3. the regularizer is pre-trained as an aliasing denoiser, so it actually
#     regularizes once ADMM gives it real weight. (Training it the old way,
#     where its effect was ~2e-4 of the data term, taught it nothing.)
# ============================================================================

# ---- full-frame references (pseudo-GT for the denoiser) ---------------------
print("computing full-frame references (CG 120)...")
ref_vols = {}
for t in NAMES:
    idx = frame_idx[null_frame[t]]
    F_p = NonuniformFourierTransformOperator(k_traj_adc[:, idx], (N_channels, N_x, N_x, N_x))
    Q_p = C_t.H @ F_p.H @ F_p @ C_t
    b_p = C_t.H @ F_p.H @ signal[:, idx].contiguous().view(torch.float32).ravel()
    x0  = torch.zeros(N_x**3, dtype=torch.complex64).view(torch.float32)
    ref_vols[t] = conjugate_gradient(Q_p, b_p, x0, maxiter=120,
                                     disable_progressbar=True).view(torch.complex64).reshape(N_x, N_x, N_x)
    print(f"  {t} done")

# ---- aliased training pairs: CG reconstructions of under-sampled subsets ----
SUB_SPOKES = (150, 100, 60)
REPS = 5
pairs_y, pairs_x = [], []
for t in tqdm(NAMES, desc="pairs (per tissue)"):
    idx_all = frame_idx[null_frame[t]]
    nsp = len(idx_all) // N_x
    for nsel in SUB_SPOKES:
        for rep in range(REPS):
            sel = np.random.RandomState(7000 + nsel * 10 + rep).permutation(nsp)[:nsel]
            idx = gather(idx_all, sel)
            F_p = NonuniformFourierTransformOperator(k_traj_adc[:, idx], (N_channels, N_x, N_x, N_x))
            d_p = signal[:, idx].contiguous().view(torch.float32).ravel()
            Q_p = C_t.H @ F_p.H @ F_p @ C_t
            b_p = C_t.H @ F_p.H @ d_p
            x0  = torch.zeros(N_x**3, dtype=torch.complex64).view(torch.float32)
            y = conjugate_gradient(Q_p, b_p, x0, maxiter=cg_iters,
                                   disable_progressbar=True).view(torch.complex64).reshape(N_x, N_x, N_x)
            pairs_y.append(y)
            pairs_x.append(ref_vols[t])
Y = torch.stack(pairs_y)[:, None]    # (N, 1, n, n, n)  aliased CG reconstructions
X = torch.stack(pairs_x)[:, None]    # (N, 1, n, n, n)  full-frame references
print(f"pairs: {tuple(Y.shape)}")

# ---- train the regularizer as an aliasing denoiser (prox-like) ---------------
# loss = MSE(denoise aliased) + 0.5 * MSE(identity on clean images);
# the second term makes the net ~identity on already-clean images, i.e. the
# behavior of a genuine proximal operator.
denoiser = ResNet(contrasts=1, features=16, num_of_resblocks=4, scale_factor=1,
                  kernel_size=(3, 3, 3), device=device, dtype=torch.complex64)
opt = torch.optim.Adam(denoiser.parameters(), lr=1e-3)
B = 6
losses = []
for phase, (n_steps, lr) in enumerate([(1200, 1e-3), (4000, 3e-4)]):
    for pg in opt.param_groups:
        pg["lr"] = lr
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_steps) if phase else None
    for step in tqdm(range(n_steps), desc=f"denoiser pretrain phase {phase+1}"):
        sel = torch.randperm(len(Y))[:B]
        pred = denoiser(Y[sel])
        loss = ((pred - X[sel]).abs() ** 2).mean() \
             + 0.5 * ((denoiser(X[sel]) - X[sel]).abs() ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        if sched is not None:
            sched.step()
        losses.append(loss.item())
print(f"final pretrain loss {losses[-1]:.3e}")

# ---- assemble the DeepADMM model ---------------------------------------------
torch.manual_seed(0)
model = DeepADMM((N_x, N_x, N_x, 1), rho_init=3e-4, maxiter_admm=1, maxiter_cg=20)
model.regularizer.regularizer.load_state_dict(denoiser.state_dict())
model.rho.requires_grad_(False)      # rho is a fixed hyperparameter here
torch.save(model.state_dict(), "deepadmm_nullframes.pt")
print("model saved to deepadmm_nullframes.pt")


In [ ]:
model.eval()

def recon_cg(idx, maxiter=cg_iters):
    'standard CG-SENSE (same operator chain as cell 17)'
    F_p = NonuniformFourierTransformOperator(k_traj_adc[:, idx], (N_channels, N_x, N_x, N_x))
    d_p = signal[:, idx].contiguous().view(torch.float32).ravel()
    Q_p = C_t.H @ F_p.H @ F_p @ C_t
    b_p = C_t.H @ F_p.H @ d_p
    x0  = torch.zeros(N_x**3, dtype=torch.complex64).view(torch.float32)
    return conjugate_gradient(Q_p, b_p, x0, maxiter=maxiter,
                              disable_progressbar=True).view(torch.complex64).reshape(N_x, N_x, N_x)

def recon_admm(idx, x0, maxiter_admm=1, maxiter_cg=20):
    'DeepADMM: normalized data term + learned regularizer, warm-started from x0'
    F_p = NonuniformFourierTransformOperator(k_traj_adc[:, idx], (N_channels, N_x, N_x, N_x))
    d_p = signal[:, idx].contiguous().view(torch.float32).ravel()
    nK  = d_p.numel() // 2
    Q_p = (C_t.H @ F_p.H @ F_p @ C_t) * (1.0 / nK)    # normalized, matches training
    b_p = (C_t.H @ F_p.H @ d_p) * (1.0 / nK)
    x0f = x0.reshape(-1).view(torch.float32).reshape(-1)   # flat real vector for the operators
    with torch.no_grad():
        x = model(Q_p, b_p, x0f,
                  maxiter_admm=maxiter_admm, maxiter_cg=maxiter_cg)
    return x.view(torch.complex64).reshape(N_x, N_x, N_x)

# ---- evaluate at several sampling levels --------------------------------------
# 750 spokes = full null frame (over-determined), 150/60 = under-determined.
EVAL_SPOKES = [N_segments, 150, 60]
results = {}
for t in NAMES:
    idx_full = frame_idx[null_frame[t]]
    nsp = len(idx_full) // N_x
    ref = ref_vols[t]
    r = ref.abs()
    m = r > 0.2 * r.max()
    rmean = r[m].mean().item()

    def err(x):
        return float(torch.abs(x.reshape(N_x, N_x, N_x) - ref)[m].mean() / rmean)

    print(f"{t} nulled (reference: full-frame CG120)")
    for nsel in EVAL_SPOKES:
        if nsel < nsp:
            sel = np.random.RandomState(9000 + nsel).permutation(nsp)[:nsel]
            idx = gather(idx_full, sel)
        else:
            idx = idx_full
        x30  = recon_cg(idx, maxiter=cg_iters)
        x200 = recon_cg(idx, maxiter=200)
        xad  = recon_admm(idx, x30)
        results[(t, nsel)] = (x30, x200, xad)
        print(f"  {nsel:4d} spokes:  CG({cg_iters})={err(x30):.4f}   CG(200)={err(x200):.4f}   DeepADMM={err(xad):.4f}")

null_admm = {t: results[(t, N_segments)][2].abs().cpu().numpy() for t in NAMES}

plt.figure(figsize=(6, 3))
plt.plot(losses, alpha=0.4)
k = 10
plt.plot(np.arange(k - 1, len(losses)), np.convolve(losses, np.ones(k) / k, 'valid'))
plt.xlabel("step"); plt.ylabel("denoiser pretrain loss"); plt.show()


In [ ]:
for name, vols in [("CG-SENSE", null_vols), ("DeepADMM", null_admm)]:
    fig, axes = plt.subplots(3, 3, figsize=(5, 5))
    for r, t in enumerate(NAMES):
        vol = vols[t]; vmax = np.percentile(vol, 99.5)
        views = [np.rot90(vol[N_x // 2].T, 2), np.rot90(vol[:, N_x // 2].T, 2),
                 np.rot90(vol[:, :, N_x // 2].T, 2)]
        for c, v in enumerate(views):
            axes[r, c].imshow(v, cmap="gray", vmin=0, vmax=vmax); axes[r, c].axis("off")
        axes[r, 0].set_title(f"{t} nulled, TI = {ti_effs[null_frame[t]]*1e3:.0f} ms", fontsize=9, loc="left")
    plt.suptitle(f"{name} - null-TI frames"); plt.tight_layout(); plt.show()